# **Explorando** IA Generativa em um Pipeline de ETL com Python

In [1]:
# Versão do Python
!python --version

Python 3.12.13


In [2]:
# Atribuir o caminho do arquivo CSV para uma variável
dados_clientes = "/content/dados_clientes.csv"

Extração

In [3]:
# Importa a biblioteca pandas para manipulação e análise de dados.
import pandas as pd

In [4]:
# Carrega o arquivo CSV "dados_clientes.csv" em um DataFrame do pandas.
df = pd.read_csv(dados_clientes)

# Extrai todos os IDs de clientes do DataFrame e os armazena em uma lista.
user_ids = df['ID'].tolist()

# Imprime a lista de IDs de usuários.
print(user_ids)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [6]:
df.head(10)

,ID,Nome,Agencia,Numero_Conta,Saldo,Limite_Conta,Numero_Cartao,Limite_Cartao,News
0,1,Ana Silva,4135,42395-2,43344.08,4807.32,4677 6003 3874 5363,7226.02,NaN
1,2,Bruno Santos,6991,12248-3,19179.37,9507.69,4506 7054 7590 7431,2783.51,NaN
2,3,Carla Oliveira,4264,11559-6,37655.27,4454.23,4653 5324 4565 2280,8173.26,NaN
3,4,Daniel Costa,3712,85197-1,17761.79,836.48,4542 7615 7385 3392,17407.67,NaN
4,5,Eduarda Pereira,5295,47435-6,5112.89,6774.85,4035 3062 8690 1701,8433.39,NaN
5,6,Felipe Rodrigues,5085,50472-1,30480.72,6477.36,4590 3818 9692 8898,19235.01,NaN
6,7,Gabriela Almeida,7539,46210-3,39666.58,3050.06,4511 5370 4523 3046,12975.95,NaN
7,8,Henrique Souza,4611,44993-7,14330.21,5220.39,4198 7248 2577 4701,15887.14,NaN
8,9,Isabela Lima,4407,83987-0,26133.85,1659.15,4828 5962 4621 5718,6497.63,NaN
9,10,João Martins,721,77041-9,12789.20,7792.77,4795 6756 3952 7266,10331.02,NaN


In [7]:
# Importa as bibliotecas necessárias para manipulação de JSON, números e avaliação literal de strings.
import json
import numpy as np
import ast # Importa o módulo 'ast' para avaliar strings com estruturas Python

# Define uma função para obter os dados de um usuário pelo ID.
def get_user(id):
  # Filtra o DataFrame para encontrar o usuário com o ID fornecido.
  user_data = df[df['ID'] == id]
  # Verifica se o usuário foi encontrado.
  if not user_data.empty:
    # Converte os dados do usuário para um dicionário.
    user_dict = user_data.iloc[0].to_dict()

    # Tenta converter a string 'News' para uma lista se ela existir e for uma string.
    if isinstance(user_dict.get('News'), str):
      try:
        user_dict['News'] = ast.literal_eval(user_dict['News'])
      except (ValueError, SyntaxError):
        # Se houver erro na avaliação ou se não for uma lista válida, inicializa como vazia.
        user_dict['News'] = []
    # Se o campo 'News' for NaN (ausente) ou ainda não for uma lista, inicializa como uma lista vazia.
    elif pd.isna(user_dict.get('News')) or not isinstance(user_dict.get('News'), list):
      user_dict['News'] = []
    return user_dict
  # Retorna None se o usuário não for encontrado.
  return None

# Cria uma lista de dicionários, onde cada dicionário contém os dados de um usuário.
# A função get_user é chamada para cada ID, e usuários válidos são adicionados à lista.
users_data = [user for id in user_ids if (user := get_user(id)) is not None]

# Imprime os dados de todos os usuários em formato JSON indentado para melhor legibilidade.
print(json.dumps(users_data, indent=2))

[
  {
    "ID": 1,
    "Nome": "Ana Silva",
    "Agencia": 4135,
    "Numero_Conta": "42395-2",
    "Saldo": 43344.08,
    "Limite_Conta": 4807.32,
    "Numero_Cartao": "4677 6003 3874 5363",
    "Limite_Cartao": 7226.02,
    "News": []
  },
  {
    "ID": 2,
    "Nome": "Bruno Santos",
    "Agencia": 6991,
    "Numero_Conta": "12248-3",
    "Saldo": 19179.37,
    "Limite_Conta": 9507.69,
    "Numero_Cartao": "4506 7054 7590 7431",
    "Limite_Cartao": 2783.51,
    "News": []
  },
  {
    "ID": 3,
    "Nome": "Carla Oliveira",
    "Agencia": 4264,
    "Numero_Conta": "11559-6",
    "Saldo": 37655.27,
    "Limite_Conta": 4454.23,
    "Numero_Cartao": "4653 5324 4565 2280",
    "Limite_Cartao": 8173.26,
    "News": []
  },
  {
    "ID": 4,
    "Nome": "Daniel Costa",
    "Agencia": 3712,
    "Numero_Conta": "85197-1",
    "Saldo": 17761.79,
    "Limite_Conta": 836.48,
    "Numero_Cartao": "4542 7615 7385 3392",
    "Limite_Cartao": 17407.67,
    "News": []
  },
  {
    "ID": 5,
    "Nome"

Transformação

In [8]:
# Importa a biblioteca google.generativeai para interagir com o Gemini API.
import google.generativeai as genai
# Importa a função userdata do google.colab para acessar segredos.
from google.colab import userdata

# Obtém a chave da API Gemini armazenada nos segredos do Colab.
GOOGLE_API_KEY=userdata.get('api_key_gemini')
# Configura a API Gemini com a chave obtida.
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [9]:
# Lista todos os modelos disponíveis na API Gemini.
for m in genai.list_models():
  # Verifica se o modelo suporta o método 'generateContent' e, se sim, imprime seu nome.
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025

In [10]:
# Inicializa o modelo generativo 'gemma-3-1b-it' da API Gemini.
gemini_model = genai.GenerativeModel('gemma-3-1b-it')

In [11]:
# Define uma função para gerar uma mensagem de notícia personalizada usando o modelo Gemini.
def generate_ai_news(user_info):

  # Cria a mensagem de prompt para o modelo, incluindo o nome do usuário e o tópico.
  prompt_message = f"Crie uma mensagem para {user_info['Nome']} sobre a importância dos investimentos (máximo de 100 caracteres)"

  # Gera conteúdo (notícia) usando o modelo Gemini com o prompt definido.
  response = gemini_model.generate_content(prompt_message)

  # Extrai o texto da resposta do modelo e remove as aspas extras, se houver.
  responseChatGPT = response.text.strip('"')
  return responseChatGPT

# Itera sobre cada item (usuário) na lista de dados dos usuários.
for user_item in users_data:
  # Gera uma notícia de IA para o usuário atual.
  news = generate_ai_news(user_item)
  # Imprime a notícia gerada.
  print(news)

  # Adiciona a notícia gerada à lista de notícias do usuário, com um ícone padrão.
  user_item['News'].append({
      "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
      "description": news
  })

Olá Ana! 🚀 Invista no futuro! Pequenos passos, grandes resultados. 😉
Bruno, investir é plantar futuro! 💰 Acredite no crescimento, o sucesso te espera! 😉
Carla, investir é plantar futuro! 💰 Pequenos passos, grandes resultados. 😉
Daniel, invista no futuro! Pequenos investimentos, grandes resultados. 😉
Eduarda, invista no futuro! Pequenos investimentos, grandes resultados. 😉
Felipe, invista no futuro! 🚀 Pequenos passos, grandes resultados. 😉
Gabriela, invista no futuro! Pequenos investimentos, grandes resultados. 😉 #investimentos #futuro
Olá, Henrique! Invista no futuro! Pequenos investimentos, grandes resultados. 😉
Isabela, invista no futuro! Pequenos investimentos, grandes resultados. 😉
João, investir é plantar futuro! Pequenos passos, grandes resultados. 😉


In [12]:
# Imprime os dados atualizados de todos os usuários em formato JSON indentado para melhor legibilidade.
print(json.dumps(users_data, indent=2))

[
  {
    "ID": 1,
    "Nome": "Ana Silva",
    "Agencia": 4135,
    "Numero_Conta": "42395-2",
    "Saldo": 43344.08,
    "Limite_Conta": 4807.32,
    "Numero_Cartao": "4677 6003 3874 5363",
    "Limite_Cartao": 7226.02,
    "News": [
      {
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": "Ol\u00e1 Ana! \ud83d\ude80 Invista no futuro! Pequenos passos, grandes resultados. \ud83d\ude09"
      }
    ]
  },
  {
    "ID": 2,
    "Nome": "Bruno Santos",
    "Agencia": 6991,
    "Numero_Conta": "12248-3",
    "Saldo": 19179.37,
    "Limite_Conta": 9507.69,
    "Numero_Cartao": "4506 7054 7590 7431",
    "Limite_Cartao": 2783.51,
    "News": [
      {
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": "Bruno, investir \u00e9 plantar futuro! \ud83d\udcb0 Acredite no crescimento, o sucesso te espera! \ud83d\ude09"
      }
    ]
  },
  {
    "I

Carga

In [14]:
# Cria um novo DataFrame com os IDs dos usuários e as notícias atualizadas.
# Isso é feito transformando a lista users_data em uma lista de dicionários para o DataFrame.
updated_news_df = pd.DataFrame([{'ID': user['ID'], 'News': user['News']} for user in users_data])

# Mescla o DataFrame original (df) com o DataFrame de notícias atualizadas (updated_news_df).
# A mesclagem é feita com base na coluna 'ID', usando um 'left merge' para manter todos os usuários do df original.
# O sufixo '' é usado para evitar conflitos de nomes de colunas, e os nomes de colunas '_old' são para indicar que são os dados antigos.
df = df.merge(updated_news_df, on='ID', how='left', suffixes=('_old', ''))

# Verifica se a coluna 'News_old' existe (que seria a coluna 'News' original antes da mesclagem).
if 'News_old' in df.columns:
    # Se existir, remove a coluna 'News_old' para manter apenas a coluna 'News' atualizada.
    df = df.drop(columns=['News_old'])

# Exibe as primeiras 5 linhas do DataFrame resultante para verificar as atualizações.
df.head()

,ID,Nome,Agencia,Numero_Conta,Saldo,Limite_Conta,Numero_Cartao,Limite_Cartao,News
0,1,Ana Silva,4135,42395-2,43344.08,4807.32,4677 6003 3874 5363,7226.02,[{'icon': 'https://digitalinnovationone.github...
1,2,Bruno Santos,6991,12248-3,19179.37,9507.69,4506 7054 7590 7431,2783.51,[{'icon': 'https://digitalinnovationone.github...
2,3,Carla Oliveira,4264,11559-6,37655.27,4454.23,4653 5324 4565 2280,8173.26,[{'icon': 'https://digitalinnovationone.github...
3,4,Daniel Costa,3712,85197-1,17761.79,836.48,4542 7615 7385 3392,17407.67,[{'icon': 'https://digitalinnovationone.github...
4,5,Eduarda Pereira,5295,47435-6,5112.89,6774.85,4035 3062 8690 1701,8433.39,[{'icon': 'https://digitalinnovationone.github...


In [15]:
# Define o caminho onde o novo arquivo CSV com os dados atualizados dos clientes será salvo.
dados_clientes_atualizado = "/content/dados_clientes_atualizado.csv"
# Imprime o caminho completo onde o arquivo será salvo para confirmação.
print(f"Novo arquivo de clientes atualizados será salvo em: {dados_clientes_atualizado}")

Novo arquivo de clientes atualizados será salvo em: /content/dados_clientes_atualizado.csv


In [16]:
# Salva o DataFrame 'df' (com os dados dos clientes atualizados) em um novo arquivo CSV.
# 'index=False' impede que o índice do DataFrame seja salvo como uma coluna no CSV.
df.to_csv(dados_clientes_atualizado, index=False)

# Imprime uma mensagem de sucesso, confirmando que os dados foram salvos e informando o caminho do arquivo.
print(f"Dados atualizados com sucesso em {dados_clientes_atualizado}")

Dados atualizados com sucesso em /content/dados_clientes_atualizado.csv


In [18]:
# Carregar o arquivo CSV atualizado de volta em um DataFrame para verificação.
df_verificacao = pd.read_csv(dados_clientes_atualizado)

# Exibir as primeiras 5 linhas do DataFrame carregado para confirmar que os dados foram salvos corretamente.
display(df_verificacao.head())

,ID,Nome,Agencia,Numero_Conta,Saldo,Limite_Conta,Numero_Cartao,Limite_Cartao,News
0,1,Ana Silva,4135,42395-2,43344.08,4807.32,4677 6003 3874 5363,7226.02,[{'icon': 'https://digitalinnovationone.github...
1,2,Bruno Santos,6991,12248-3,19179.37,9507.69,4506 7054 7590 7431,2783.51,[{'icon': 'https://digitalinnovationone.github...
2,3,Carla Oliveira,4264,11559-6,37655.27,4454.23,4653 5324 4565 2280,8173.26,[{'icon': 'https://digitalinnovationone.github...
3,4,Daniel Costa,3712,85197-1,17761.79,836.48,4542 7615 7385 3392,17407.67,[{'icon': 'https://digitalinnovationone.github...
4,5,Eduarda Pereira,5295,47435-6,5112.89,6774.85,4035 3062 8690 1701,8433.39,[{'icon': 'https://digitalinnovationone.github...
